In [19]:
import glob
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
import rioxarray
from rasterio.warp import Resampling, reproject
from shapely.geometry import box

# --------------------------------------------------
# Choose one event
# --------------------------------------------------

ens = "01"
year = 1990
month = 12
day = 1

# Use an event from your dataframe if easier:
# row = rainfall_events_all_df.iloc[0]
# ens = row["ens_num"]
# year = int(row["start_year"])
# month = int(row["start_month"])
# day = int(row["start_day"])


# --------------------------------------------------
# Paths
# --------------------------------------------------

sm_dir = (f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_{ens}/")
sm_dir = (f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_{ens}")
ep_file = "../../Data/SoilStaticVariables/Mean_EffectivePorosity_GB_30m.nc"

dem_file = "../../Data/Model_builds/Pluvial/v4/dem/dem_Dolwen.tif"


# --------------------------------------------------
# Find soil-moisture file
# --------------------------------------------------

sm_file = glob.glob(
    f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc"
)[0]

print("Soil moisture file:")
print(sm_file)


# --------------------------------------------------
# Load soil moisture
# --------------------------------------------------

ds = xr.open_dataset(sm_file)

sm = ds["moisture_content_of_soil_layer"]



Soil moisture file:
/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_01/r001i1p00000_19891201-19901130_mrso.nc


In [24]:
sm_day = sm.isel(time=0)

ep_30m = xr.open_dataset(
    ep_file
)["effective_porosity"]

In [30]:
dem_file = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/Model_builds/Pluvial/v4/dem/dem_filled_30m_49.tif"
with rasterio.open(dem_file) as dem:

    dem_transform = dem.transform
    dem_crs = dem.crs
    dem_width = dem.width
    dem_height = dem.height
    dem_bounds = dem.bounds
    dem_mask = dem.read(1)

print(dem_width, dem_height)
print(dem_crs)

4809 3764
EPSG:27700


In [32]:
sm_day = sm_day.rio.write_crs("EPSG:27700")

sm_clipped = sm_day.rio.clip_box(
    minx=dem_bounds.left,
    miny=dem_bounds.bottom,
    maxx=dem_bounds.right,
    maxy=dem_bounds.top,
)

print(sm_clipped)

source_array = np.squeeze(sm_clipped.values)

sm_30m = np.empty(
    (dem_height, dem_width),
    dtype=source_array.dtype
)

reproject(
    source=source_array,
    destination=sm_30m,
    src_transform=sm_clipped.rio.transform(),
    src_crs=sm_clipped.rio.crs,
    dst_transform=dem_transform,
    dst_crs=dem_crs,
    resampling=Resampling.nearest,
)

<xarray.DataArray 'moisture_content_of_soil_layer' (
                                                    projection_y_coordinate: 23,
                                                    projection_x_coordinate: 30)> Size: 6kB
[690 values with dtype=float64]
Coordinates:
    time                     object 8B 1989-12-01 12:00:00
  * projection_y_coordinate  (projection_y_coordinate) float64 184B 7.5e+03 ....
  * projection_x_coordinate  (projection_x_coordinate) float64 240B 8.25e+04 ...
    transverse_mercator      int64 8B 0
Attributes:
    standard_name:  moisture_content_of_soil_layer
    units:          kg m-2


(array([[ 0.        ,  0.        ,  0.        , ..., 86.16075897,
         86.16075897, 86.16075897],
        [ 0.        ,  0.        ,  0.        , ..., 86.16075897,
         86.16075897, 86.16075897],
        [ 0.        ,  0.        ,  0.        , ..., 86.16075897,
         86.16075897, 86.16075897],
        ...,
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ]], shape=(3764, 4809)),
 Affine(30.0, 0.0, 82710.0,
        0.0, -30.0, 118230.0))

In [36]:
sm_30m

array([[ 0.        ,  0.        ,  0.        , ..., 86.16075897,
        86.16075897, 86.16075897],
       [ 0.        ,  0.        ,  0.        , ..., 86.16075897,
        86.16075897, 86.16075897],
       [ 0.        ,  0.        ,  0.        , ..., 86.16075897,
        86.16075897, 86.16075897],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(3764, 4809))

In [35]:
# vol_sm = ((grav_data / 1000.0) / 0.225) / porosity_data
# vol_sm = np.clip(vol_sm, 0, 1)

raw_saturation = (
    (sm_30m / 1000.0) / 0.225
) / ep_30m

ValueError: operands could not be broadcast together with shapes (3764,4809) (35034,21670) 

In [1]:
### think about units
### how to apply to events?

In [2]:
# import xarray as xr
# import matplotlib.pyplot as plt
# nc_path ="../../Data/SoilDynamicVariables/combined_from_tif/Ens_01/nc/23/soil_23_2_1994_7_11_Ens_01_vol_5km.nc"
# hc_5km = xr.open_dataset(nc_path, engine="netcdf4")

# import os

# print(nc_path)
# print(os.path.exists(nc_path))
# print(os.path.getsize(nc_path))
# plt.pcolormesh(hc_5km['remaining_capacity'])

# hc_5km['remaining_capacity'].data


In [3]:
# import glob
# import numpy as np
# import xarray as xr

# # ------------------------------------------------------------
# # Hydraulic conductivity → effective soil depth
# # ------------------------------------------------------------

# hc_5km = xr.open_dataset("../../Data/HydraulicConductivity/GB_5km.nc")

# # Extract hydraulic conductivity DataArray
# Ks = hc_5km["aggregated_mean"]

# # Calculate effective soil depth
# L_5km = 4 * np.sqrt(Ks)

# # Rename variable
# L_5km = L_5km.rename("effective_soil_depth")

# # Set units
# L_5km.attrs["units"] = "m"

# # Convert to mm
# L_5km_mm = L_5km * 1000


# # ------------------------------------------------------------
# # Effective porosity
# # ------------------------------------------------------------

# ep_5km = xr.open_dataset( "../../Data/EffectivePorosity/GB_5km_mean.nc")["aggregated_mean"]


# # # ------------------------------------------------------------
# # # Load soil moisture
# # # ------------------------------------------------------------

# # ens = "01"
# # year = 1990

# # sm_dir = (f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/"f"Ens_{ens}/")

# # sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc")[0]

# # sm_ds = xr.open_dataset(sm_file)

# # mrso = sm_ds["moisture_content_of_soil_layer"]

# # # mrso is kg/m², which is equivalent to mm of water
# # water_depth_mm = mrso


# # # ------------------------------------------------------------
# # # Volumetric water content
# # # ------------------------------------------------------------

# # vol_water_content = water_depth_mm / L_5km_mm


# # # ------------------------------------------------------------
# # # Degree of saturation
# # # ------------------------------------------------------------

# # saturation = vol_water_content / ep_5km


# # # ------------------------------------------------------------
# # # Physical limits
# # # ------------------------------------------------------------

# # saturation = saturation.where(saturation >= 0)
# # saturation = saturation.clip(max=1)


# # # ------------------------------------------------------------
# # # Maximum infiltration volume
# # # ------------------------------------------------------------

# # max_infiltration_vol = ep_5km * L_5km_mm


# # # ------------------------------------------------------------
# # # Remaining infiltration volume
# # # ------------------------------------------------------------

# # remain_infiltration_vol = saturation * max_infiltration_vol


KeyboardInterrupt



In [1]:
# output_dir = Path("../../Data/Soil_saturation/saturation_5km/")

# output_dir.mkdir(parents=True, exist_ok=True)

# soil_depth_mm = L_5km.copy()
# soil_depth_mm.data = L_5km.data * 1000

In [2]:
# effective_porosity = iris.load("../../Data/EffectivePorosity/GB_5km_mean.nc")[0]
# effective_porosity = xr.open_dataset("../../Data/EffectivePorosity/GB_5km_mean.nc", engine="netcdf4")
# sm_cube = xr.open_dataset(sm_file)
# soil_depth_mm = 225

In [ ]:
# ens='01'
# sm_dir = (f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_{ens}/")
# year=1990
# soil_depth_mm = 225

# output_file = output_dir / f"soil_saturation_Ens_{ens}_{year}.nc"

# # Skip if already processed
# # if output_file.exists():
# #     continue

# sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc")[0]


# # --------------------------------------------------
# # Load one year
# # --------------------------------------------------

# sm_cube = xr.open_dataset(sm_file)

# # --------------------------------------------------
# # Convert masked -> NaN
# # --------------------------------------------------

# if np.ma.is_masked(sm_cube['moisture_content_of_soil_layer']):
#     sm_data = sm_cube['moisture_content_of_soil_layer'].filled(np.nan)
# else:
#     sm_data = sm_cube['moisture_content_of_soil_layer']

# # --------------------------------------------------
# # Water depth
# # kg/m2 == mm
# # --------------------------------------------------

# water_depth_mm = sm_data

# # --------------------------------------------------
# # Volumetric water content
# # --------------------------------------------------

# theta = (water_depth_mm /soil_depth_mm)

# # --------------------------------------------------
# # Saturation
# # --------------------------------------------------

# saturation = (theta /effective_porosity['aggregated_mean'])

# # --------------------------------------------------
# # Keep physically sensible values
# # --------------------------------------------------

# #         saturation = np.where(saturation < 0,np.nan,saturation)
# #         saturation = np.where(saturation > 1,1,saturation)

# saturation = saturation.where(saturation >= 0)
# saturation = saturation.clip(max=1)

# # --------------------------------------------------
# # Max storage depth
# # --------------------------------------------------
# max_storage_depth = effective_porosity['aggregated_mean'] * L_5km_mm

# remaining_storage_depth = ((1 - saturation) * max_storage_depth)


# # --------------------------------------------------
# # Put result into cube
# # --------------------------------------------------

# sat_cube = sm_cube.copy(data=saturation.astype(np.float32))

# sat_cube.rename("degree_of_saturation")
# sat_cube.units = "1"

# # --------------------------------------------------
# # Save
# # --------------------------------------------------
# # print(output_file)
# # iris.save(sat_cube,str(output_file))


In [7]:
import iris
import iris.cube
import numpy as np
import glob
from pathlib import Path
from tqdm import tqdm
import xarray as xr
import matplotlib.pyplot as plt

In [49]:
hc_5km = xr.open_dataset("../../Data/SoilStaticVariables/Mean_HydraulicConductivity_GB_5km.nc")["aggregated_mean"]
lu_5km = xr.open_dataset("../../Data/SoilStaticVariables/Mean_EffectiveDepth_GB_5km.nc")["aggregated_mean"]
ep_5km = xr.open_dataset("../../Data/SoilStaticVariables/Mean_EffectivePorosity_GB_5km.nc")["aggregated_mean"]
fumax_5km = xr.open_dataset("../../Data/SoilStaticVariables/Mean_FUmax_GB_5km.nc")["aggregated_mean"]
mode_texture_5km = xr.open_dataset("../../Data/SoilStaticVariables/Mode_SoilTexture_GB_5km.nc")["mode"]

In [102]:
# ep_30m = xr.open_dataset("../../Data/SoilStaticVariables/Mean_EffectivePorosity_GB_30m.nc")["effective_porosity"]
ep_30m

<xarray.DataArray 'effective_porosity' (y: 35034, x: 21670)> Size: 3GB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(35034, 21670), dtype=float32)
Coordinates:
  * x        (x) float64 173kB 5.535e+03 5.565e+03 ... 6.556e+05 6.556e+05
  * y        (y) float64 280kB 1.056e+06 1.056e+06 ... 5.355e+03 5.325e+03
    band     int64 8B ...
Attributes:
    AREA_OR_POINT:  Area
    units:          cm3 cm-3
    grid_mapping:   spatial_ref

In [115]:
saturation_raw = water_depth_mm / (soil_depth_mm * ep_5km)

print("Raw maximum:", saturation_raw.max().values)
print("Raw mean:", saturation_raw.mean().values)

print(
    "Fraction > 1:",
    (saturation_raw > 1).mean().values
)

print(
    "Fraction > 1:",
    (saturation_raw > 1).sum().values /
    saturation_raw.notnull().sum().values
)

print(
    saturation_raw.quantile(
        [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1]
    ).values
)

raw_saturation = (
    (water_depth_mm / 1000)
    / 0.225
    / ep_5km
)

exceeding = raw_saturation > 1

print("Number of values > 1:", exceeding.sum().values)
print("Total valid values:", raw_saturation.notnull().sum().values)

print(
    "Percentage > 1:",
    (exceeding.sum() / raw_saturation.notnull().sum() * 100).values
)

fraction_over_1_by_time = (saturation_raw > 1).mean(
    dim=["projection_y_coordinate", "projection_x_coordinate"]
)

print(fraction_over_1_by_time.mean().values)
print(fraction_over_1_by_time.min().values)
print(fraction_over_1_by_time.max().values)

0.07224960787290022
0.019922586520947177
0.12056010928961748

Raw maximum: 13.84865073746169
Raw mean: 0.8570671502926537
Fraction > 1: 0.07224960787290022
Fraction > 1: 0.3099133487428243
[ 0.          0.          0.          0.53331613  0.72555477  0.87383889
  1.04611359  1.18921135  1.28182435  1.48100631 13.84865074]
Number of values > 1: 1142353
Total valid values: 3686040
Percentage > 1: 30.99133487428243
0.07224960787290022
0.019922586520947177
0.12056010928961748


In [106]:
import rioxarray as rxr

DIR = "/scratch/hydro5/users/la17355/FUTURE-FLOOD/UKCP_soil_moisture/volumetric_30m/soil_depth_missing_tests/Ens_01/tif/23/"
soil_file = DIR + f"soil_23_95_2080_8_7_Ens_01_vol_30m.tif" 

# Get soil texture data 
soil_texture = rxr.open_rasterio(soil_file).squeeze() 
# avoid dask chunks here; merge wants computed arrays 
soil_texture = soil_texture.where(soil_texture != -9999) 
soil_values = soil_texture.values 

np.nanmax(soil_values)

np.float32(1.0)

In [50]:
output_dir = Path("../../Data/Soil_saturation/saturation_5km/")
output_dir.mkdir(parents=True, exist_ok=True)

In [55]:
soil_depth_mm = xr.full_like(hc_5km, 225.0).where(hc_5km.notnull())
soil_depth_mm.name ='soil_depth_mm'

In [57]:
# Loop through ensemble members
for ens in ['01', '04']:

    print(f"\nProcessing ensemble {ens}")
    
    # define directory where 5km soil moisture is stored
    sm_dir = (f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_{ens}/")
    
    # Loop through years
    for year in tqdm(range(1980, 1982), desc=f"Ens {ens}"):
        
        # define where output will be saved
        output_file = output_dir / f"soil_saturation_Ens_{ens}_{year}.nc"

        # Skip if already processed
        if output_file.exists():
            continue
            
        # --------------------------------------------------
        # Load 5km soil moisture in kg/m2 for one year
        # --------------------------------------------------
        try:
            sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc")[0]
        except IndexError:
            continue
        sm_cube = xr.open_dataset(sm_file)

        # --------------------------------------------------
        # Convert masked -> NaN
        # --------------------------------------------------

        if np.ma.is_masked(sm_cube['moisture_content_of_soil_layer']):
            sm_data = sm_cube['moisture_content_of_soil_layer'].filled(np.nan)
        else:
            sm_data = sm_cube['moisture_content_of_soil_layer']

        # --------------------------------------------------
        # Convert to water depth
        # kg/m2 == mm
        # --------------------------------------------------

        water_depth_mm = sm_data

        # --------------------------------------------------
        # Convert to volumetric water content
        # water_depth_mm is 3D and soil_depth_mm is 2D,
        # But xarray should automatically broadcast the soil depth over the 360 time steps.
        # --------------------------------------------------

        theta = water_depth_mm / soil_depth_mm
        theta = theta.rename("vol_water_content")
        
        # --------------------------------------------------
        # Saturation
        # --------------------------------------------------

        saturation = theta / ep_5km
        saturation = saturation.rename("soil_saturation")
        
#         # --------------------------------------------------
#         # Keep physically sensible values
#         # --------------------------------------------------

# #         saturation = np.where(saturation < 0,np.nan,saturation)
# #         saturation = np.where(saturation > 1,1,saturation)

#         saturation = saturation.where(saturation >= 0)
#         saturation = saturation.clip(max=1)
        
#         # --------------------------------------------------
#         # Max storage depth
#         # --------------------------------------------------
#         max_storage_depth = effective_porosity['aggregated_mean'] * L_5km_mm

#         remaining_storage_depth = ((1 - saturation) * max_storage_depth)
        
        
#         # --------------------------------------------------
#         # Put result into cube
#         # --------------------------------------------------

#         sat_cube = sm_cube.copy(data=saturation.astype(np.float32))

#         sat_cube.rename("degree_of_saturation")
#         sat_cube.units = "1"

#         # --------------------------------------------------
#         # Save
#         # --------------------------------------------------
#         print(output_file)
#         iris.save(sat_cube,str(output_file))

#         # Explicitly release memory
#         del sm_cube
#         del sm_data
#         del water_depth_mm
#         del theta
#         del saturation
#         del sat_cube


Processing ensemble 01


Ens 01: 100%|██████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.78it/s]



Processing ensemble 04


Ens 04: 100%|██████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.98s/it]


In [92]:
iris.load('/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_04/r001i1p01113_19801201-19811130_mrso.nc')[0]

<iris 'Cube' of moisture_content_of_soil_layer / (kg m-2) (time: 360; projection_y_coordinate: 244; projection_x_coordinate: 180)>

In [86]:
# saturation_ds = saturation.to_dataset()
# output_ds = xr.Dataset({"vol_water_content": theta, "soil_saturation": saturation})
print(water_depth_mm.min().values)
print(water_depth_mm.max().values)

print(soil_depth_mm.min().values)
print(soil_depth_mm.max().values)

print(ep_5km.min().values)
print(ep_5km.max().values)

max_storage_mm = soil_depth_mm * ep_5km

print("Max possible storage:")
print(max_storage_mm.min().values)
print(max_storage_mm.max().values)

print("Met Office soil moisture:")
print(water_depth_mm.min().values)
print(water_depth_mm.max().values)

ratio = water_depth_mm / (soil_depth_mm * ep_5km)

print("Maximum saturation:", ratio.max().values)
print("Mean saturation:", ratio.mean().values)

0.0
115.55902099609375
225.0
225.0
0.0
0.48538796794585926
Max possible storage:
0.043312498927116395
109.21229278781833
Met Office soil moisture:
0.0
115.55902099609375
Maximum saturation: 13.84865073746169
Mean saturation: 0.8570671502926537


In [88]:
print(
    ep_5km.where(ep_5km < 0.05).count().values
)
print(ep_5km.quantile([0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99, 1]).values)

153
[0.         0.00596421 0.24946139 0.31101971 0.34099396 0.39384903
 0.41439945 0.43237417 0.46770342 0.48538797]


In [23]:
ens = '01'
year = 1990
print(f"\nProcessing ensemble {ens}")

# define directory where 5km soil moisture is stored
sm_dir = (f"/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_{ens}/")


# define where output will be saved
output_file = output_dir / f"soil_saturation_Ens_{ens}_{year}.nc"

# Skip if already processed
# if output_file.exists():
#     continue

# --------------------------------------------------
# Load 5km soil moisture in kg/m2 for one year
# --------------------------------------------------
# try:
#     sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc")[0]
# except IndexError:
#     continue
sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc")[0]
sm_cube = xr.open_dataset(sm_file)

# --------------------------------------------------
# Convert masked -> NaN
# --------------------------------------------------

if np.ma.is_masked(sm_cube['moisture_content_of_soil_layer']):
    sm_data = sm_cube['moisture_content_of_soil_layer'].filled(np.nan)
else:
    sm_data = sm_cube['moisture_content_of_soil_layer']

# --------------------------------------------------
# Convert to water depth
# kg/m2 == mm
# --------------------------------------------------

water_depth_mm = sm_data

# --------------------------------------------------
# Convert to volumetric water content
# --------------------------------------------------

theta = water_depth_mm / soil_depth_mm

# --------------------------------------------------
# Saturation
# --------------------------------------------------

saturation = (theta /effective_porosity['aggregated_mean'])

# --------------------------------------------------
# Keep physically sensible values
# --------------------------------------------------

#         saturation = np.where(saturation < 0,np.nan,saturation)
#         saturation = np.where(saturation > 1,1,saturation)

saturation = saturation.where(saturation >= 0)
saturation = saturation.clip(max=1)

# --------------------------------------------------
# Max storage depth
# --------------------------------------------------
max_storage_depth = effective_porosity['aggregated_mean'] * L_5km_mm

remaining_storage_depth = ((1 - saturation) * max_storage_depth)


# --------------------------------------------------
# Put result into cube
# --------------------------------------------------

sat_cube = sm_cube.copy(data=saturation.astype(np.float32))

sat_cube.rename("degree_of_saturation")
sat_cube.units = "1"

# --------------------------------------------------
# Save
# --------------------------------------------------
print(output_file)
iris.save(sat_cube,str(output_file))



Processing ensemble 01


NameError: name 'effective_porosity' is not defined

In [22]:
soil_depth_mm['soil_depth_mm'] is projection_y_coordinate: 244projection_x_coordinate: 180

<xarray.DataArray 'soil_depth_mm' (projection_y_coordinate: 244,
                                   projection_x_coordinate: 180)> Size: 351kB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(244, 180))
Coordinates:
  * projection_x_coordinate  (projection_x_coordinate) float32 720B -1.975e+0...
  * projection_y_coordinate  (projection_y_coordinate) float32 976B -3.25e+04...

In [20]:
water_depth_mm 

<xarray.DataArray 'moisture_content_of_soil_layer' (time: 360,
                                                    projection_y_coordinate: 244,
                                                    projection_x_coordinate: 180)> Size: 126MB
[15811200 values with dtype=float64]
Coordinates:
  * time                     (time) object 3kB 1989-12-01 12:00:00 ... 1990-1...
  * projection_y_coordinate  (projection_y_coordinate) float64 2kB -3.25e+04 ...
  * projection_x_coordinate  (projection_x_coordinate) float64 1kB -1.975e+05...
Attributes:
    standard_name:  moisture_content_of_soil_layer
    units:          kg m-2
    grid_mapping:   transverse_mercator